# Conformal prediction

Paper-facing tables for the 90% conformal prediction experiments. The analysis uses the linear probe, excludes TENX143 and Hibou-L, retains only LEMON ViT-S/8 under the label LEMON, and labels the four-layer, all-training-cells, adapted CellBridge representation simply as CellBridge.

The first table compares representations under branch graph, the proposed conformal construction. The second fixes the CellBridge representation and crosses four calibration scopes (global, class, branch, and hybrid) with two set constructions (split and graph). Global graph is the closest analogue of the graph-structured method proposed by Corbetta et al. HEST (ID) and all OOD benchmarks appear in the same table.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))
CELLBRIDGE_RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_4L_INFERENCE_RESULTS",
    WORKSPACE / "outputs/ablations/lora_depth/layers_4_cells_all/inference",
))

PROBE = "linear"
EXCLUDED_HEST_SAMPLES = {"TENX143"}
METHOD_COMPARISON_CP = "branch_graph"

METHOD_GROUPS = [
    ("Patch-level FMs", ["UNI2-h", "Virchow2"]),
    ("Cell-level FMs", ["VOLTA", "LEMON"]),
    ("Multi-modal FMs", ["PAST", "SpatialWhisperer"]),
    ("Ours", ["CellBridge"]),
]
METHOD_ORDER = [
    method
    for _, methods in METHOD_GROUPS
    for method in methods
]

CONFORMAL_GROUPS = [
    ("Global calibration", ["split", "graph"]),
    ("Class calibration", ["class_split", "class_graph"]),
    ("Branch calibration", ["branch_split", "branch_graph"]),
    ("Hybrid calibration", ["hybrid_split", "hybrid_graph"]),
]
ABLATION_ORDER = [
    method
    for _, methods in CONFORMAL_GROUPS
    for method in methods
]
CONFORMAL_LABELS = {
    "split": "Global split",
    "graph": "Global graph (Corbetta et al.)",
    "class_split": "Class split",
    "class_graph": "Class graph",
    "branch_split": "Branch split",
    "branch_graph": "Branch graph (ours)",
    "hybrid_split": "Hybrid split",
    "hybrid_graph": "Hybrid graph",
}

BENCHMARK_ORDER = [
    "HEST",
    "NuCLS",
    "PanNuke",
    "Lizard",
    "PhenoCell (all)",
    "PhenoCell (immune)",
]
BENCHMARK_LABELS = {
    "HEST": "HEST (ID)",
    "Lizard": "Lizard (OOD)",
    "NuCLS": "NuCLS (OOD)",
    "PanNuke": "PanNuke (OOD)",
    "PhenoCell (all)": "PhenoCell all (OOD)",
    "PhenoCell (immune)": "PhenoCell immune (OOD)",
}

TABLE_METRICS = [
    ("marginal_coverage", "Marginal coverage"),
    ("mean_set_size", "Mean set size (↓)"),
    ("singleton_rate", "Singleton rate (↑)"),
]
COVGAP_TABLE_METRICS = [("coverage_gap", "CovGap (↓)")]
COVGAP_ABLATION_ORDER = ["split", "graph", "branch_graph"]
LATEX_METRIC_LABELS = {
    "marginal_coverage": r"Marginal coverage",
    "mean_set_size": r"Mean set size ($\downarrow$)",
    "singleton_rate": r"Singleton rate ($\uparrow$)",
    "coverage_gap": r"CovGap ($\downarrow$)",
}
METRIC_ABBREVIATIONS = {
    "marginal_coverage": "MC",
    "mean_set_size": "SS",
    "singleton_rate": "SR",
}

ROW_COLOUR = "gray!15"
OURS_BACKGROUND = "#E6E6E6"

## Load and aggregate the conformal metrics

HEST entries are means across the 14 retained tissue sections; each OOD entry comes from its fixed held-out test split. Marginal coverage and singleton rate are reported as percentages. Mean set size is the mean number of labels retained per prediction set.


In [ ]:
BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}


def paper_method(method, version, arm):
    if method == "cellbridge" or arm.startswith("lora_"):
        size = version.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return "CellBridge {} ({})".format(size, representation)
    return BASELINE_LABELS.get(method, method)


def benchmark_label(dataset, task):
    if dataset == "phenocell":
        return "PhenoCell ({})".format(task)
    return {
        "hest": "HEST",
        "lizard": "Lizard",
        "nucls": "NuCLS",
        "pannuke": "PanNuke",
    }[dataset]


def load_conformal_metrics(root, cellbridge_root):
    if not cellbridge_root.is_dir():
        raise FileNotFoundError(
            "Four-layer CellBridge results do not exist: {}".format(cellbridge_root)
        )
    paths = list(root.glob(
        "*/*/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(root.glob(
        "*/*/hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    # The four-layer run has dataset directories directly below its root.
    paths += list(cellbridge_root.glob(
        "*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(cellbridge_root.glob(
        "hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    tables = []
    replaced_results, replacement_results = set(), set()
    required = {
        "method",
        "arm",
        "task",
        "probe",
        "conformal_method",
        "target_coverage",
        "marginal_coverage",
        "mean_prediction_set_size",
        "singleton_rate",
    }
    for path in sorted(paths):
        from_four_layer = path.is_relative_to(cellbridge_root)
        if from_four_layer:
            # Prefix the implied method/version so both layouts parse alike.
            parts = ("cellbridge", "cells_all") + path.relative_to(cellbridge_root).parts
        else:
            parts = path.relative_to(root).parts
        method, version, dataset = parts[:3]
        if dataset == "hest":
            sample_id = parts[3]
            arm, task, probe = parts[5:8]
        else:
            sample_id = pd.NA
            arm, task, probe = parts[4:7]

        # CellBridge all/adapted comes only from the four-layer run; the
        # two-layer cells_all/lora_adapted results are replaced, never a fallback.
        result_key = (dataset, str(sample_id), task, probe)
        if from_four_layer:
            if arm != "lora_adapted":
                continue
            replacement_results.add(result_key)
        elif method == "cellbridge" and version == "cells_all" and arm == "lora_adapted":
            if str(sample_id) not in EXCLUDED_HEST_SAMPLES:
                replaced_results.add(result_key)
            continue

        table = pd.read_csv(path)
        missing = required.difference(table.columns)
        if missing:
            raise ValueError(
                "{} is missing columns: {}".format(path, sorted(missing))
            )
        table["version"] = version
        table["sample_id"] = sample_id
        table["paper_method"] = paper_method(method, version, arm)
        table["benchmark"] = benchmark_label(dataset, task)
        table["result_dir"] = str(path.parent)
        tables.append(table)

    missing_replacements = replaced_results - replacement_results
    if missing_replacements:
        raise FileNotFoundError(
            "Four-layer CellBridge results are missing for: {}".format(
                sorted(missing_replacements)
            )
        )
    if not tables:
        raise FileNotFoundError(
            "No conformal metrics found below {}".format(root)
        )

    table = pd.concat(tables, ignore_index=True)
    table = table.loc[table["method"] != "hibou_l"]
    table = table.loc[
        ~((table["method"] == "lemon") & (table["version"] != "vits8"))
    ]
    table = table.loc[
        ~table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)
    ]
    return table.reset_index(drop=True)


def aggregate_points(table):
    return (
        table.groupby(
            ["paper_method", "benchmark", "probe", "conformal_method"],
            as_index=False,
        )
        .agg(
            marginal_coverage=("marginal_coverage", "mean"),
            mean_set_size=("mean_prediction_set_size", "mean"),
            singleton_rate=("singleton_rate", "mean"),
            coverage_gap=("coverage_gap", "mean"),
            target_coverage=("target_coverage", "mean"),
            n_hest_sections=("sample_id", "count"),
        )
    )


conformal = load_conformal_metrics(RESULTS_ROOT, CELLBRIDGE_RESULTS_ROOT)
points = aggregate_points(conformal)

target_coverages = conformal["target_coverage"].dropna().unique()
if len(target_coverages) != 1:
    raise ValueError(
        "Expected one nominal coverage target, found {}".format(
            sorted(target_coverages)
        )
    )
NOMINAL_COVERAGE = float(target_coverages[0])

print(
    "Loaded {:,} conformal rows at {:.0%} nominal coverage; "
    "probe = {}; excluded HEST = {}".format(
        len(conformal),
        NOMINAL_COVERAGE,
        PROBE,
        sorted(EXCLUDED_HEST_SAMPLES),
    )
)
print(
    "CellBridge = four-layer all/adapted results from {}".format(
        CELLBRIDGE_RESULTS_ROOT
    )
)

## Representation comparison under branch graph

This table compares all baseline representations with CellBridge using branch graph and the linear probe. Methods are grouped by representation family. The displayed table stacks the three metrics beneath each method; the LaTeX table gives each method one row, with marginal coverage (MC), mean set size (SS), and singleton rate (SR) side by side under each dataset. Dataset columns are grouped into ID and OOD. Gray shading on CellBridge identifies our proposed representation and conformal construction.


In [ ]:
def _format_metric(value, metric):
    if pd.isna(value):
        return "--"
    if metric in {"marginal_coverage", "singleton_rate"}:
        return "{:.1f}".format(100 * value)
    if metric == "coverage_gap":
        return "{:.1f}".format(value)
    return "{:.2f}".format(value)


TABLE_BENCHMARK_LABELS = {
    "HEST": "HEST",
    "Lizard": "Lizard",
    "NuCLS": "NuCLS",
    "PanNuke": "PanNuke",
    "PhenoCell (all)": "PhenoCell all",
    "PhenoCell (immune)": "PhenoCell immune",
}
LATEX_BENCHMARK_LABELS = {
    "PhenoCell all": r"PhenoCell\textsubscript{all}",
    "PhenoCell immune": r"PhenoCell\textsubscript{imm}",
}
BENCHMARK_SPLITS = {
    benchmark: "ID" if benchmark == "HEST" else "OOD"
    for benchmark in BENCHMARK_ORDER
}


def build_rowwise_table(
    rows,
    row_key,
    row_order,
    row_heading,
    row_labels=None,
    metrics=None,
):
    metrics = metrics or TABLE_METRICS
    duplicate = rows.duplicated([row_key, "benchmark"], keep=False)
    if duplicate.any():
        raise ValueError(
            "Duplicate table entries:\n{}".format(
                rows.loc[
                    duplicate,
                    [row_key, "benchmark"],
                ].to_string(index=False)
            )
        )

    observed = set(
        rows[[row_key, "benchmark"]].itertuples(index=False, name=None)
    )
    missing = [
        (row, benchmark)
        for row in row_order
        for benchmark in BENCHMARK_ORDER
        if (row, benchmark) not in observed
    ]
    if missing:
        raise ValueError("Missing table entries: {}".format(missing))

    metric_keys = [metric for metric, _ in metrics]
    raw = rows.pivot(
        index=row_key,
        columns="benchmark",
        values=metric_keys,
    )
    labels = row_labels or {}
    values = []
    index = []
    for row in row_order:
        row_label = labels.get(row, row)
        for metric, metric_label in metrics:
            index.append((row_label, metric_label))
            values.append([
                _format_metric(raw.loc[row, (metric, benchmark)], metric)
                for benchmark in BENCHMARK_ORDER
            ])

    columns = pd.MultiIndex.from_tuples(
        [
            (
                BENCHMARK_SPLITS[benchmark],
                TABLE_BENCHMARK_LABELS[benchmark],
            )
            for benchmark in BENCHMARK_ORDER
        ],
        names=["Split", "Dataset"],
    )
    return pd.DataFrame(
        values,
        index=pd.MultiIndex.from_tuples(
            index,
            names=[row_heading, "Metric"],
        ),
        columns=columns,
    )


def rank_table_cells(rows, row_key, row_order, row_labels=None, metrics=None):
    labels = row_labels or {}
    metrics = metrics or TABLE_METRICS
    lookup = rows.set_index([row_key, "benchmark"])
    emphasis = {"best": set(), "second": set()}

    for metric, metric_label in metrics:
        for benchmark in BENCHMARK_ORDER:
            candidates = []
            for row in row_order:
                value = lookup.loc[(row, benchmark), metric]
                if pd.isna(value):
                    continue
                displayed_value = float(_format_metric(value, metric))
                if metric == "marginal_coverage":
                    score = round(abs(displayed_value - 90.0), 10)
                elif metric in {"mean_set_size", "coverage_gap"}:
                    score = displayed_value
                else:
                    score = -displayed_value
                candidates.append((row, score))

            score_levels = sorted({score for _, score in candidates})
            for rank, score_level in zip(
                ("best", "second"),
                score_levels[:2],
            ):
                for row, score in candidates:
                    if score == score_level:
                        emphasis[rank].add((
                            labels.get(row, row),
                            metric_label,
                            TABLE_BENCHMARK_LABELS[benchmark],
                        ))
    return emphasis


def display_table(table, emphasis, shaded_rows, label_level):
    def shade(row):
        row_label = row.name[label_level]
        colour = (
            "background-color: {}".format(OURS_BACKGROUND)
            if row_label in shaded_rows
            else ""
        )
        return [colour] * len(row)

    def emphasize(data):
        styles = pd.DataFrame("", index=data.index, columns=data.columns)
        for index in data.index:
            row_label = index[label_level]
            metric_label = index[-1]
            for column in data.columns:
                key = (row_label, metric_label, column[-1])
                if key in emphasis["best"]:
                    styles.at[index, column] = "font-weight: bold"
                elif key in emphasis["second"]:
                    styles.at[index, column] = "text-decoration: underline"
        return styles

    display(
        table.style
        .apply(shade, axis=1)
        .apply(emphasize, axis=None)
        .set_properties(**{"text-align": "center"})
        .set_table_styles([
            {
                "selector": "th.row_heading.level{}".format(label_level),
                "props": [
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                ],
            }
        ])
    )


def _header_segments(columns, level):
    segments = []
    start = 0
    for position in range(1, len(columns) + 1):
        if (
            position == len(columns)
            or columns[position][level] != columns[start][level]
        ):
            segments.append((columns[start][level], start, position - 1))
            start = position
    return segments


def emit_latex_table(
    table,
    caption,
    label,
    emphasis,
    shaded_rows,
    label_level,
    row_heading,
    grouped=False,
    metrics=None,
):
    metrics = metrics or TABLE_METRICS
    total_columns = 2 + len(table.columns)
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\setlength{\tabcolsep}{4pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll" + "c" * len(table.columns) + "}",
        r"\toprule",
    ]

    split_headers = []
    split_rules = []
    for split, start, end in _header_segments(table.columns, level=0):
        split_headers.append(
            r"\multicolumn{"
            + str(end - start + 1)
            + r"}{c}{\textbf{"
            + str(split)
            + "}}"
        )
        split_rules.append(
            r"\cmidrule(lr){"
            + "{}-{}".format(start + 3, end + 3)
            + "}"
        )
    lines.append(" & & " + " & ".join(split_headers) + r" \\")
    lines.append(" ".join(split_rules))
    lines.append(
        row_heading
        + " & Metric & "
        + " & ".join(
            LATEX_BENCHMARK_LABELS.get(column[-1], column[-1])
            for column in table.columns
        )
        + r" \\"
    )
    lines.append(r"\midrule")

    latex_metric_labels = {
        display_label: LATEX_METRIC_LABELS[metric]
        for metric, display_label in metrics
    }
    previous_group = None
    previous_row = None
    have_rows = False
    for index, row in table.iterrows():
        group = index[0] if grouped else None
        row_label = index[label_level]
        metric_label = index[-1]

        if grouped and group != previous_group:
            if have_rows:
                lines.append(r"\midrule")
            lines.append(
                r"\multicolumn{"
                + str(total_columns)
                + r"}{l}{\textbf{"
                + str(group)
                + r"}} \\"
            )
            lines.append(r"\addlinespace")
            previous_row = None
        elif row_label != previous_row and have_rows:
            lines.append(r"\addlinespace")

        prefix = (
            r"\rowcolor{" + ROW_COLOUR + "} "
            if row_label in shaded_rows
            else ""
        )
        is_first_metric = row_label != previous_row
        is_last_metric = metric_label == metrics[-1][1]
        if row_label in shaded_rows:
            shown_row_label = (
                r"\multirow{-"
                + str(len(metrics))
                + r"}{*}{"
                + str(row_label)
                + "}"
                if is_last_metric
                else ""
            )
        else:
            shown_row_label = (
                r"\multirow{"
                + str(len(metrics))
                + r"}{*}{"
                + str(row_label)
                + "}"
                if is_first_metric
                else ""
            )
        formatted_values = []
        for column, value in row.items():
            key = (row_label, metric_label, column[-1])
            shown_value = str(value)
            if key in emphasis["best"]:
                shown_value = r"\textbf{" + shown_value + "}"
            elif key in emphasis["second"]:
                shown_value = r"\underline{" + shown_value + "}"
            formatted_values.append(shown_value)
        lines.append(
            prefix
            + shown_row_label
            + " & "
            + latex_metric_labels[metric_label]
            + " & "
            + " & ".join(formatted_values)
            + r" \\"
        )
        previous_group = group
        previous_row = row_label
        have_rows = True

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"}",
        r"\end{table*}",
    ])
    print("\n".join(lines))


def emit_wide_latex_table(table, caption, label, emphasis, shaded_rows, metrics=None):
    """Print one row per method, with the metrics side by side under each dataset."""
    metrics = metrics or TABLE_METRICS
    width = len(metrics)
    benchmarks = [column[-1] for column in table.columns]

    def data_columns(first, last):
        return "{}-{}".format(2 + width * first, 1 + width * (last + 1))

    def align(position, count):
        return "c" if position == count - 1 else "c|"

    lines = [
        r"% Requires booktabs, graphicx, and \usepackage[table]{xcolor}",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\setlength{\tabcolsep}{3pt}",
        r"\renewcommand{\arraystretch}{1.08}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l|" + "|".join(["c" * width] * len(benchmarks)) + "}",
        r"\toprule",
    ]
    segments = _header_segments(table.columns, level=0)
    for position, (split, start, end) in enumerate(segments):
        lines.append(
            r" & \multicolumn{" + str(width * (end - start + 1)) + "}{"
            + align(position, len(segments)) + r"}{\textbf{" + str(split) + "}}"
        )
    lines[-1] += r" \\"
    for split, start, end in segments:
        lines.append(r"\cmidrule(lr){" + data_columns(start, end) + "}")
    for position, benchmark in enumerate(benchmarks):
        lines.append(
            r" & \multicolumn{" + str(width) + "}{" + align(position, len(benchmarks))
            + "}{" + LATEX_BENCHMARK_LABELS.get(benchmark, benchmark) + "}"
        )
    lines[-1] += r" \\"
    for position in range(len(benchmarks)):
        lines.append(r"\cmidrule(lr){" + data_columns(position, position) + "}")
    lines.append(r"\textbf{Method}")
    abbreviations = " & ".join(METRIC_ABBREVIATIONS[metric] for metric, _ in metrics)
    lines.extend(" & " + abbreviations for _ in benchmarks)
    lines[-1] += r" \\"

    metric_labels = [metric_label for _, metric_label in metrics]
    for family in dict.fromkeys(index[0] for index in table.index):
        lines.append(r"\midrule")
        lines.append(r"\textbf{" + str(family) + "}")
        lines.extend(
            r" & \multicolumn{" + str(width) + "}{" + align(position, len(benchmarks)) + "}{}"
            for position in range(len(benchmarks))
        )
        lines[-1] += r" \\"
        for method in dict.fromkeys(index[1] for index in table.index if index[0] == family):
            if method in shaded_rows:
                lines.append(r"\rowcolor{" + ROW_COLOUR + "}")
                lines.append(r"\textbf{" + str(method) + "}")
            else:
                lines.append(str(method))
            for column in table.columns:
                cells = []
                for metric_label in metric_labels:
                    value = str(table.loc[(family, method, metric_label), column])
                    key = (method, metric_label, column[-1])
                    if key in emphasis["best"]:
                        value = r"\textbf{" + value + "}"
                    elif key in emphasis["second"]:
                        value = r"\underline{" + value + "}"
                    cells.append(value)
                lines.append(" & " + " & ".join(cells))
            lines[-1] += r" \\"
    lines.extend([r"\bottomrule", r"\end{tabular}%", "}", r"\end{table*}"])
    print("\n".join(lines))


main_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()
main_table = build_rowwise_table(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
    row_heading="Method",
)
family_by_method = {
    method: family
    for family, methods in METHOD_GROUPS
    for method in methods
}
main_table.index = pd.MultiIndex.from_tuples(
    [
        (family_by_method[method], method, metric)
        for method, metric in main_table.index
    ],
    names=["Family", "Method", "Metric"],
)

main_emphasis = rank_table_cells(
    main_rows,
    row_key="paper_method",
    row_order=METHOD_ORDER,
)
main_shaded_rows = {"CellBridge"}
display_table(
    main_table,
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
    label_level=1,
)
emit_wide_latex_table(
    main_table,
    caption=(
        "Representation comparison under branch-conditional graph conformal\n"
        "prediction using linear probes. MC: marginal coverage (\\%);\n"
        "SS: mean set size ($\\downarrow$); SR: singleton rate (\\%, $\\uparrow$).\n"
        "The nominal coverage target is {:.0f}\\%. HEST entries average 14 held-out\n"
        "sections. The gray row identifies CellBridge."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-representation-comparison",
    emphasis=main_emphasis,
    shaded_rows=main_shaded_rows,
)

## Conformal efficiency by dataset

One figure per benchmark task compares the seven representations under the selected
PROBE and METHOD_COMPARISON_CP (linear / branch graph by default). The x-axis is
mean set size and the y-axis is singleton rate. Marginal coverage remains in the
tables above and is not annotated on these figures.

Colors, Computer Modern fonts, LaTeX rendering, and axis styling match notebook07.
Figures are narrower and taller (6 by 6.5 inches), with methods listed from top to
bottom in a single-column legend inside the upper-right corner.
HEST points average the 14 retained sections; external points use the fixed test
splits. PhenoCell all and immune are plotted separately. Vector PDFs are saved under
outputs/reproducibility/figures/conformal_prediction by default.


In [ ]:
import shutil
import subprocess

import matplotlib.pyplot as plt

# Match the TeX setup and plotting conventions in notebook07.
def _ensure_tex():
    missing = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
    if missing:
        modulecmd = shutil.which("modulecmd")
        if modulecmd is None:
            raise RuntimeError("Cannot load texlive: modulecmd is unavailable")
        command = subprocess.run(
            [modulecmd, "python", "load", "texlive-20220321"],
            check=True, capture_output=True, text=True,
        )
        exec(command.stdout, {"os": os})
    missing = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
    if missing:
        raise RuntimeError(
            f"TeX executables remain unavailable: {', '.join(missing)}"
        )

_ensure_tex()
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["cm10"],
    "font.size": 18,
    "axes.edgecolor": "darkgray",
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linestyle": "--",
    "grid.color": "lightgray",
    "figure.dpi": 120,
    "savefig.dpi": 300,
})
plt.rc("text.latex", preamble=r"\usepackage{amsmath,bm}")

PLOT_METHOD_LABELS = {"SpatialWhisperer": "SW"}
METHOD_COLORS = {
    "UNI2-h": {"linear": "#CB722F", "knn": "#DEBA9B"},
    "Virchow2": {"linear": "#93300C", "knn": "#C99886"},
    "VOLTA": {"linear": "#BD5673", "knn": "#EAB5C4"},
    "LEMON": {"linear": "#D6A238", "knn": "#EBD19C"},
    "PAST": {"linear": "#8859AB", "knn": "#C0ADD2"},
    "SpatialWhisperer": {"linear": "#287DC5", "knn": "#9CBDDF"},
    "CellBridge": {"linear": "#88C2AB", "knn": "#C9E0D6"},
}

FIGURE_DIR = Path(os.environ.get(
    "CELLBRIDGE_CONFORMAL_FIGURES",
    WORKSPACE / "outputs/reproducibility/figures/conformal_prediction",
))
SAVE_FIGURES = True
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PLOT_BENCHMARKS = {
    "HEST": ("HEST", "hest"),
    "NuCLS": ("NuCLS", "nucls"),
    "PanNuke": ("PanNuke", "pannuke"),
    "Lizard": ("Lizard", "lizard"),
    "PhenoCell (all)": (r"PhenoCell$_{\mathrm{all}}$", "phenocell_all"),
    "PhenoCell (immune)": (r"PhenoCell$_{\mathrm{imm}}$", "phenocell_immune"),
}
scatter_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()


def plot_conformal_efficiency(benchmark):
    rows = (
        scatter_rows.loc[scatter_rows["benchmark"] == benchmark]
        .set_index("paper_method")
        .loc[METHOD_ORDER]
    )
    title, slug = PLOT_BENCHMARKS[benchmark]
    fig, ax = plt.subplots(figsize=(6, 6.5))
    for method, row in rows.iterrows():
        ax.scatter(
            row["mean_set_size"],
            row["singleton_rate"],
            s=100,
            color=METHOD_COLORS[method][PROBE],
            edgecolor="#222222",
            linewidth=0.75,
            alpha=0.92,
            label=PLOT_METHOD_LABELS.get(method, method),
            zorder=3,
        )

    x = rows["mean_set_size"]
    y = rows["singleton_rate"]
    x_padding = max(0.08, 0.12 * (x.max() - x.min()))
    y_padding = max(0.02, 0.30 * (y.max() - y.min()))
    ax.set_xlim(max(0, x.min() - x_padding), x.max() + x_padding)
    ax.set_ylim(max(0, y.min() - y_padding), min(1, y.max() + y_padding))
    ax.set_xlabel(r"Mean set size")
    ax.set_ylabel(r"Singleton rate (\%)")
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda value, _: f"{100 * value:.0f}")
    )
    ax.set_title(title)
    ax.set_axisbelow(True)
    ax.legend(
        ncol=1,
        frameon=False,
        loc="upper right",
        fontsize=14,
    )
    fig.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(
            FIGURE_DIR / f"fig_conformal_{slug}_{PROBE}_{METHOD_COMPARISON_CP}_efficiency.pdf",
            bbox_inches="tight",
        )
    plt.show()
    return fig


efficiency_figures = {
    benchmark: plot_conformal_efficiency(benchmark)
    for benchmark in BENCHMARK_ORDER
}


## CellBridge conformal-prediction ablation

The table crosses all four calibration scopes with both set constructions implemented in the repository. Every configuration uses the same four-layer CellBridge representation and linear probe; each has three metric rows.

- **Global calibration:** one threshold is estimated from all calibration examples.
- **Class calibration:** one threshold is estimated per true class.
- **Branch calibration:** one threshold is estimated per configured Cell Ontology branch.
- **Hybrid calibration:** class-specific thresholds are used when calibration support is sufficient; an entire branch falls back to one branch threshold if it contains a low-support class.
- **Split construction:** candidate labels are thresholded using the standard nonconformity score, one minus the predicted probability.
- **Graph construction:** prediction sets use graph-inclusion scores and are closed to the Cell Ontology task graph. Global graph is the closest analogue of Corbetta et al.'s graph-structured method; the conditionally calibrated graph variants are extensions.

The three gray rows identify the proposed branch-graph configuration.


In [ ]:
ablation_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["paper_method"] == "CellBridge")
    & points["conformal_method"].isin(ABLATION_ORDER)
].copy()
ablation_table = build_rowwise_table(
    ablation_rows,
    row_key="conformal_method",
    row_order=ABLATION_ORDER,
    row_heading="Configuration",
    row_labels=CONFORMAL_LABELS,
)
calibration_by_label = {
    CONFORMAL_LABELS[method]: calibration
    for calibration, methods in CONFORMAL_GROUPS
    for method in methods
}
ablation_table.index = pd.MultiIndex.from_tuples(
    [
        (calibration_by_label[configuration], configuration, metric)
        for configuration, metric in ablation_table.index
    ],
    names=["Calibration", "Configuration", "Metric"],
)

ablation_emphasis = rank_table_cells(
    ablation_rows,
    row_key="conformal_method",
    row_order=ABLATION_ORDER,
    row_labels=CONFORMAL_LABELS,
)
proposed_configuration = {"Branch graph (ours)"}
display_table(
    ablation_table,
    emphasis=ablation_emphasis,
    shaded_rows=proposed_configuration,
    label_level=1,
)
emit_latex_table(
    ablation_table,
    caption=(
        "CellBridge conformal-prediction ablation using linear probes. "
        "The table crosses global, class, branch, and hybrid calibration "
        "with split and graph set constructions. Global graph is the closest "
        "analogue of Corbetta et al. Marginal coverage and singleton rate are "
        "percentages. The nominal {:.0f}\\% "
        "coverage target should be considered together with mean set "
        "size, and singleton rate. HEST is ID and all other benchmarks are "
        "OOD. Gray rows identify branch graph."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-construction-ablation",
    emphasis=ablation_emphasis,
    shaded_rows=proposed_configuration,
    label_level=1,
    row_heading="Configuration",
    grouped=True,
)


## CellBridge conformal-prediction ablation: CovGap

Global split, global graph, and branch graph from the ablation above, reporting the class-conditional coverage gap (CovGap; Ding et al., 2023): the mean absolute deviation of per-class coverage from the nominal target, in percentage points, averaged over the classes present in each test set. Lower is better, and 0 means every class is covered at exactly the target. Because the deviation is absolute, over-coverage of a class counts as much as under-coverage. HEST entries average the per-section CovGap over the 14 retained sections. Best and second-best are ranked among these three rows only.

The gray row identifies the proposed branch-graph configuration.


In [ ]:
covgap_table = build_rowwise_table(
    ablation_rows,
    row_key="conformal_method",
    row_order=COVGAP_ABLATION_ORDER,
    row_heading="Configuration",
    row_labels=CONFORMAL_LABELS,
    metrics=COVGAP_TABLE_METRICS,
)
covgap_table.index = pd.MultiIndex.from_tuples(
    [
        (calibration_by_label[configuration], configuration, metric)
        for configuration, metric in covgap_table.index
    ],
    names=["Calibration", "Configuration", "Metric"],
)

covgap_emphasis = rank_table_cells(
    ablation_rows,
    row_key="conformal_method",
    row_order=COVGAP_ABLATION_ORDER,
    row_labels=CONFORMAL_LABELS,
    metrics=COVGAP_TABLE_METRICS,
)
display_table(
    covgap_table,
    emphasis=covgap_emphasis,
    shaded_rows=proposed_configuration,
    label_level=1,
)
emit_latex_table(
    covgap_table,
    caption=(
        "CellBridge CovGap for global split, global graph (the closest "
        "analogue of Corbetta et al.), and branch graph using linear "
        "probes. CovGap is the mean absolute deviation of "
        "class-conditional coverage from the nominal {:.0f}\\% target, in "
        "percentage points (lower is better). HEST is ID and all other "
        "benchmarks are OOD. HEST entries average 14 retained sections. "
        "The gray row identifies branch graph."
    ).format(100 * NOMINAL_COVERAGE),
    label="tab:conformal-covgap-ablation",
    emphasis=covgap_emphasis,
    shaded_rows=proposed_configuration,
    label_level=1,
    row_heading="Configuration",
    grouped=True,
    metrics=COVGAP_TABLE_METRICS,
)

## Reading the tables

- Marginal coverage is a validity criterion, with 90% as the target rather than a quantity to maximize. Values below the target indicate undercoverage; values far above it are conservative.
- Within each dataset and metric, the best displayed value is bold and the second-best is underlined. Marginal coverage is ranked by absolute distance to 90.0, mean set size from low to high, singleton rate from high to low, and CovGap from low to high. Ties at the displayed precision receive the same emphasis.
- Mean set size and singleton rate are complementary efficiency measures. Smaller sets and more singleton predictions are desirable only while coverage remains adequate.
- HEST is the ID benchmark and is averaged across the 14 retained tissue sections. NuCLS, PanNuke, Lizard, PhenoCell (all), and PhenoCell (immune) are OOD benchmarks.
- The main table shades CellBridge (its three metric rows when displayed, one row in LaTeX). The ablation shades the three branch-graph metric rows, and the CovGap table shades the branch-graph row.
